# पुनरावर्ती न्यूरल नेटवर्क

मागील मॉड्यूलमध्ये, आम्ही मजकुराच्या समृद्ध सिमँटिक प्रतिनिधित्वांचा आढावा घेतला. आपण वापरत असलेली आर्किटेक्चर वाक्यातील शब्दांचे एकूण अर्थ जपते, पण ती शब्दांची **क्रम** लक्षात घेत नाही, कारण एम्बेडिंगनंतरची संकलन प्रक्रिया मूळ मजकुरातील ही माहिती काढून टाकते. या मॉडेल्सना शब्दांच्या क्रमाचे प्रतिनिधित्व करण्याची क्षमता नसल्यामुळे, ते मजकूर निर्मिती किंवा प्रश्नोत्तरे यांसारख्या अधिक जटिल किंवा अभिप्रेत कार्ये सोडवू शकत नाहीत.

मजकूर अनुक्रमाचा अर्थ पकडण्यासाठी, आपण **पुनरावर्ती न्यूरल नेटवर्क** किंवा RNN नावाची न्यूरल नेटवर्क आर्किटेक्चर वापरणार आहोत. RNN वापरताना, आपण आपले वाक्य टोकननंतर टोकन नेटवर्कमधून पाठवतो, आणि नेटवर्क काही **स्थिती** निर्माण करते, जी नंतर पुढील टोकनसह नेटवर्कला परत पाठवली जाते.

![पुनरावर्ती न्यूरल नेटवर्क जनरेशनचे उदाहरण दर्शवणारी प्रतिमा.](../../../../../translated_images/mr/rnn.27f5c29c53d727b5.webp)

दिलेल्या टोकन अनुक्रम $X_0,\dots,X_n$ साठी, RNN न्यूरल नेटवर्क ब्लॉक्सची एक अनुक्रम तयार करते, आणि बॅकप्रॉपगेशन वापरून हा अनुक्रम अखंडपणे प्रशिक्षित करते. प्रत्येक नेटवर्क ब्लॉक इनपुट म्हणून $(X_i,S_i)$ घेतो, आणि परिणाम स्वरूप $S_{i+1}$ तयार करतो. अंतिम स्थिती $S_n$ किंवा आउटपुट $Y_n$ एका रेषीय वर्गीकारकाकडे जातो ज्यामुळे निकाल तयार होतो. सर्व नेटवर्क ब्लॉक्स एकसारखे वजन वापरतात, आणि एक बॅकप्रॉपगेशन पास वापरून अखंडपणे प्रशिक्षित केले जातात.

> वर दाखवलेली आकृती पुनरावर्ती न्यूरल नेटवर्कच्या अनरोल्ड स्वरूपात (डाव्या बाजूला) आणि अधिक संकुचित पुनरावर्ती प्रतिनिधित्वात (उजव्या बाजूला) दर्शवते. हे समजणे महत्त्वाचे आहे की सर्व RNN सेल्समध्ये एकच **सामायिक वजन** असतात.

कारण स्थिती सदिश $S_0,\dots,S_n$ नेटवर्कमधून प्रवाहित होतात, RNN शब्दांमधील अनुक्रमणिका अवलंबित्व शिकू शकतो. उदाहरणार्थ, जेंव्हा शब्द *not* अनुक्रमात कुठेही येतो, तेव्हा तो स्थिती सदिशातील काही घटकांना नकार देण्यास शिकू शकतो.

आत, प्रत्येक RNN सेलमध्ये दोन वजन मॅट्रिसेस असतात: $W_H$ आणि $W_I$, आणि बायस $b$. प्रत्येक RNN टप्प्यावर, इनपुट म्हणून $X_i$ आणि स्थिती $S_i$ दिली असताना, आउटपुट स्थिती $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$ या प्रकारे गणित केली जाते, जिथे $f$ ही एक सक्रियता कार्य (प्रामुख्याने $\tanh$) आहे.

> मजकूर निर्मिती (ज्याचा आढावा आपण पुढील युनिटमध्ये घेणार आहोत) किंवा मशीन अनुवाद यांसारख्या समस्या साठी, प्रत्येक RNN टप्प्यावर काही आउटपुट मूल्य देखील हवे असते. अशा परिस्थितीत, आणखी एक मॅट्रिक्स $W_O$ असते, आणि आउटपुट अशी गणना केली जाते: $Y_i=f(W_O\times S_i+b_O)$.

पाहुयात की पुनरावर्ती न्यूरल नेटवर्क आपल्या बातम्या डेटासेट वर्गीकरणात कसे मदत करू शकतात.

> सॅंडबॉक्स पर्यावरणासाठी, आवश्यक लायब्ररी इंस्टॉल असल्याची खात्री करण्यासाठी आणि डेटा प्रीफेच करण्यासाठी खालील सेल चालवणे आवश्यक आहे. आपण स्थानिक प्रणालीवर चालवत असाल तर पुढील सेल वगळू शकता.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

मोठ्या मॉडेल्सचे प्रशिक्षण करताना, GPU मेमरीच्या वाटपाचा प्रश्न उद्भवू शकतो. तसेच आपल्याला विविध मिनिबॅच आकारांसह प्रयोग करण्याची गरज असू शकते, जेणेकरून डेटा आपल्या GPU मेमरीत बसेल आणि प्रशिक्षण पुरेसे वेगवान होईल. जर तुम्ही हा कोड तुमच्या स्वतःच्या GPU मशीनवर चालवत असाल, तर तुम्ही प्रशिक्षण जलद करण्यासाठी मिनिबॅच आकार समायोजित करण्याचा प्रयोग करू शकता.

> **टिप: ** NVidia ड्रायव्हरच्या काही आवृत्त्या प्रशिक्षणानंतर मेमरी सोडत नाहीत असे ज्ञात आहे. आम्ही या नोटबुकमध्ये अनेक उदाहरणे चालवत आहोत, आणि काही सेटअपमध्ये मेमरी संपुष्टात येऊ शकते, विशेषतः जर तुम्ही त्याच नोटबुकचा हिस्सा म्हणून स्वतःचे प्रयोग करत असाल तर. जर तुम्हाला मॉडेलचे प्रशिक्षण सुरू करताना काही विचित्र त्रुटी येत असतील, तर तुम्ही नोटबुक कर्नल पुन्हा सुरू करू इच्छित असू शकता.


In [3]:
batch_size = 16
embed_size = 64

## सोपा RNN वर्गीकारक

सोप्या RNN च्या बाबतीत, प्रत्येक पुनरावृत्ती युनिट एक सोपा रेषीय नेटवर्क आहे, जो इनपुट व्हेक्टर आणि स्टेट व्हेक्टर घेतो आणि एक नवीन स्टेट व्हेक्टर तयार करतो. Keras मध्ये, हे `SimpleRNN` लेयर द्वारे दर्शवू शकतो.

आपण थेट RNN लेयरला वन-हॉट एनकोडेड टोकन्स देऊ शकतो, परंतु त्यांच्या उच्च डायमेंशनलिटीमुळे हे चांगले नाही. म्हणून, आपण शब्द व्हेक्टरची डायमेंशनलिटी कमी करण्यासाठी एम्बेडिंग लेअर वापरणार आहोत, त्यानंतर RNN लेयर, आणि शेवटी `Dense` वर्गीकारक.

> **टीप**: जिथे डायमेंशनलिटी फार जास्त नाही, उदा. कॅरेक्टर-लेव्हल टोकनायझेशनमध्ये, तेव्हा वन-हॉट एनकोडेड टोकन्स थेट RNN सेलमध्ये देणे योग्य असू शकते.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **टीप:** येथे सोपेपणासाठी आम्ही एक अप्रशिक्षित एम्बेडिंग लेयर वापरत आहोत, परंतु चांगले परिणामांसाठी आपण मागील युनिटमध्ये वर्णन केलेल्या प्रमाणे Word2Vec वापरून एक pretrained एम्बेडिंग लेयर वापरू शकतो. हा कोड प्रीट्रेन्ड एम्बेडिंग्जसह कार्य करण्यासाठी रूपांतरित करणे तुमच्यासाठी एक चांगले सराव असेल.

आता आपण आपले RNN प्रशिक्षण देऊया. RNN सामान्यतः प्रशिक्षण देणे खूप कठीण असते, कारण RNN सेल्स अनुक्रम लांबीच्या अनुषंगाने अनरोल केल्यावर, बॅकप्रॉपॅगेशनमध्ये संबंधित स्तरांची संख्या खूप मोठी होते. त्यामुळे आपल्याला एक छोटा लर्निंग रेट निवडावा लागतो, आणि चांगले परिणाम देण्यासाठी नेटवर्क मोठ्या डेटासेटवर प्रशिक्षण द्यावे लागते. यासाठी बराच वेळ लागू शकतो, त्यामुळे GPU वापरणे प्राधान्य दिले जाते.

गोष्टी जलद करण्यासाठी, आम्ही फक्त बातम्यांच्या शीर्षकांवर RNN मॉडेलचे प्रशिक्षण देणार आहोत, वर्णन वगळून. तुम्ही वर्णनासह प्रशिक्षण देण्याचा प्रयत्न करू शकता आणि पाहू शकता की मॉडेल प्रशिक्षण देऊ शकतो का.


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **टीप** येथे अचूकता कमी असण्याची शक्यता आहे, कारण आपण फक्त बातम्यांच्या शीर्षकांवर प्रशिक्षण देत आहोत.


## Revisiting variable sequences 

Remember that the `TextVectorization` layer will automatically pad sequences of variable length in a minibatch with pad tokens. It turns out that those tokens also take part in training, and they can complicate convergence of the model.

There are several approaches we can take to minimize the amount of padding. One of them is to reorder the dataset by sequence length and group all sequences by size. This can be done using the `tf.data.experimental.bucket_by_sequence_length` function (see [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Another approach is to use **masking**. In Keras, some layers support additional input that shows which tokens should be taken into account when training. To incorporate masking into our model, we can either include a separate `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), or we can specify the `mask_zero=True` parameter of our `Embedding` layer.

> **Note**: This training will take around 5 minutes to complete one epoch on the whole dataset. Feel free to interrupt training at any time if you run out of patience. What you can also do is limit the amount of data used for training, by adding `.take(...)` clause after `ds_train` and `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


आता जेव्हा आपण मास्किंग वापरत आहोत, तेव्हा आपण संपूर्ण शीर्षके आणि वर्णनांच्या डेटासेटवर मॉडेल प्रशिक्षण देऊ शकतो.

> **नोट**: तुम्ही पाहिलं आहे का की आम्ही एका व्हेक्टरायझरचा वापर केला आहे जो न्यूज शीर्षकांवर प्रशिक्षित केला आहे, संपूर्ण लेखाच्या शरीरावर नाही? कदाचित, हे काही टोकन्स दुर्लक्षित होऊ शकतात, म्हणून व्हेक्टरायझरला पुनःप्रशिक्षित करणे चांगले आहे. तथापि, याचा प्रभाव अगदीच कमी असू शकतो, त्यामुळे सोपेपणासाठी आपण मागील पूर्व-प्रशिक्षित व्हेक्टरायझरवरच राहू.


## LSTM: लाँग शॉर्ट-टर्म मेमरी  

RNN च्या मुख्य समस्या पैकी एक आहे **व्हॅनिशिंग ग्रेडियंट्स**. RNN खूप लांब असू शकतात, आणि बॅकप्रॉपगेशन दरम्यान ग्रेडियंट्स प्रथम लेयरपर्यंत पोहोचवणे कठीण होते. जेव्हा असे होते, तेव्हा नेटवर्क दूरच्या टोकन्समधील संबंध शिकू शकत नाही. या समस्येपासून वाचण्यासाठी एक मार्ग आहे **स्पष्ट स्टेट मॅनेजमेंट** **गेट्स** वापरून लागू करणे. गेट्स परिचयात्मक दोन सामान्य आर्किटेक्चर्स आहेत: **लाँग शॉर्ट-टर्म मेमरी** (LSTM) आणि **गेटेड रिले युनिट** (GRU). येथे आपण LSTM पाहणार आहोत.  

![लाँग शॉर्ट टर्म मेमरी सेलचे उदाहरण दर्शवितेले प्रतिमा](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)  

LSTM नेटवर्क RNN प्रमाणेच आयोजित केलेले असते, पण दोन स्टेट्स लेयरपासून लेयरपर्यंत पाठवल्या जातात: वास्तविक स्टेट $c$, आणि हिडन व्हेक्टर $h$. प्रत्येक युनिटवर, हिडन व्हेक्टर $h_{t-1}$ इनपुट $x_t$ बरोबर एकत्र करून ते स्टेट $c_t$ आणि आउटपुट $h_{t}$ सह **गेट्स** द्वारे नियंत्रित करतात. प्रत्येक गेटला सिग्मॉइड अॅक्टिव्हेशन असते (आउटपुट श्रेणी $[0,1]$ मध्ये), ज्याला स्टेट व्हेक्टरवर गुणाकार करताना बिटवाइज मास्क म्हणून समजावे. LSTM मध्ये खालील गेट्स आहेत (व आतील उजव्या कडून डावीकडे चित्रात दिलेले):  
* **फॉरगेट गेट** ज्याद्वारे $c_{t-1}$ व्हेक्टरचे कोणते घटक विसरावे आणि कोणते पुढे पाठवायचे हे ठरते.  
* **इनपुट गेट** जे ठरवते की इनपुट व्हेक्टर आणि मागील हिडन व्हेक्टरमधून किती माहिती स्टेट व्हेक्टरमध्ये समाविष्ट करायची.  
* **आउटपुट गेट** जे नवीन स्टेट व्हेक्टर घेते आणि त्याच्या कोणत्या घटकांचा वापर नवीन हिडन व्हेक्टर $h_t$ तयार करण्यासाठी होईल ते ठरवते.  

स्टेट $c$ चे घटक म्हणजे त्या झेंडेसारखे जे ऑन-ऑफ करता येतात. उदाहरणादाखल, *Alice* हे नाव आल्यावर आपण गृहित धरतो की तो स्त्री संदर्भित करतो, आणि त्या स्टेटमध्ये झेंडा उंचावतो जो दर्शवतो की वाक्यात स्त्रीलिंगी नाम आहे. पुढे *and Tom* शब्द आल्यावर, बहुवचन नाम असल्याचे झेंडा उंचावतो. अशा प्रकारे स्टेट मॅनिप्युलेट करून आपण वाक्याच्या व्याकरणाच्या गुणधर्मांचे पालन करू शकतो.  

> **टीप**: LSTM च्या आतील रचनेची समज मिळवण्यासाठी हा उत्कृष्ट स्रोत आहे: [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) क्रिस्टोफर ओलाह यांनी लिहिलेले.  

LSTM सेलची अंतर्गत रचना जरी गुंतागुंतीची वाटली, तरी Keras या अंमलबजावणीला `LSTM` लेयरमध्ये लपवते, त्यामुळे वर दिलेल्या उदाहरणात फक्त पुनरावर्ती लेयरला बदला लागतो:  


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **टीप** की LSTM चे प्रशिक्षण देखील खूप हळू होते, आणि तुम्हाला प्रशिक्षणाच्या सुरुवातीस अचूकतेत फारसा वाढ दिसू न शकते. चांगली अचूकता मिळवण्यासाठी तुम्हाला काही काळ प्रशिक्षण चालू ठेवावे लागू शकते.


## द्विदिश आणि बहुस्तरीय RNNs

आतापर्यंतच्या आपल्या उदाहरणांत, पुनरावर्ती नेटवर्क साखळीतल्या सुरुवातीपासून शेवटपर्यंत कार्य करतात. हे नैसर्गिक वाटते कारण ते त्याच दिशेनुसार चालते ज्यामध्ये आपण वाचतो किंवा भाषण ऐकतो. मात्र, जेव्हा इनपुट साखळीमध्ये रँडम प्रवेश आवश्यक असतो, तेव्हा पुनरावर्ती संगणना दोन्ही दिशांनी धावणे अधिक अर्थपूर्ण असते. अशा संगणनांना दोन्ही दिशांत करण्यास अनुमती देणाऱ्या RNNsना **द्विदिशात्मक** RNNs म्हणतात, व त्या पुनरावर्ती थराला खास `Bidirectonal` थर घेऊन तयार करता येतात.

> **टीप**: `Bidirectional` थर त्यातील थराची दोन प्रती तयार करतो, आणि त्यातील एका प्रतीचा `go_backwards` वैशिष्ट्य `True` वर सेट करतो, ज्यामुळे ती साखळीच्या विरुद्ध दिशेने चालते.

पुनरावर्ती नेटवर्क्स, एकदिशात्मक असोत किंवा द्विदिशात्मक, साखळीतील नमुने पकडतात आणि त्यांना स्थिती सदिशात साठवतात किंवा आउटपुट म्हणून परत देतात. कन्फोल्यूशनल नेटवर्कसारखे, आपण प्रथम थरानंतर दुसरा पुनरावर्ती थर तयार करू शकतो ज्यामुळे उच्च-स्तरीय नमुने पकडले जातात, जे पहिल्या थराने काढलेले नीच-स्तरीय नमुन्यांवर आधारित असतात. यामुळे आपल्याला **बहु-स्तरीय RNN** या संकल्पनेपर्यंत पोहोचते, ज्यामध्ये दोन किंवा अधिक पुनरावर्ती नेटवर्क्स असतात, जेथे मागील थराचा आउटपुट पुढील थराला इनपुट म्हणून दिला जातो.

![इमेज दाखवत आहे बहुस्तरीय दीर्घ-अल्पकालीन-स्मृती- RNN](../../../../../translated_images/mr/multi-layer-lstm.dd975e29bb2a59fe.webp)

*चित्र [हा सुंदर पोस्ट](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) फर्नांडो लोpez यांनी तयार केलेला.*

Keras मध्ये ही नेटवर्क्स तयार करणे सोपे आहे कारण आपण फक्त मॉडेलमध्ये अधिक पुनरावर्ती थर जोडायचे असतात. शेवटच्या थराव्यतिरिक्त सर्व थरांसाठी, आपल्याला `return_sequences=True` पॅरामीटर निर्दिष्ट करावा लागतो, कारण आपल्याला थराने सर्व मधल्या स्थिती परत कराव्यात लागत आहेत, फक्त पुनरावर्ती संगणनेचे अंतिम स्थिती नव्हे.

चला आपल्या वर्गीकरण समस्येसाठी द्विदिशात्मक LSTM चे दोन-स्तरीय मॉडेल बनवू या.

> **टीप** हा कोड पुन्हा पूर्ण होण्यासाठी बराच वेळ घेतो, पण तो आतापर्यंत पाहिलेली सर्वात जास्त अचूकता देतो. म्हणून कदाचित थोडा वाट पाहणे आणि निकाल पाहणे फायदेशीर ठरेल.


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## इतर कामांसाठी RNNs

आतापर्यंत, आम्ही मजकूराची मालिका वर्गीकृत करण्यासाठी RNNs वापरण्यावर लक्ष केंद्रित केले आहे. पण ते अनेक इतर कामे देखील हाताळू शकतात, जसे की मजकूर निर्मिती आणि मशीन अनुवाद — आपण ते काम पुढील युनिटमध्ये पाहणार आहोत.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
